# AVG Training

Jalankan `Run All` satu kali dari folder tugas dengan kernel envGPU. Notebook mencari parameter CNN, melatih dan menyimpan CNN terbaik (maksimal 150 epoch), mengekstrak embedding sementara di memori untuk mencari parameter dan melatih SVM, lalu menyimpan SVM. Setelah selesai, jalankan `AVG Extraction.ipynb` untuk menyimpan embedding ke berkas `.npz`. Data official test tidak digunakan.

## 1. Konfigurasi
Tetapkan lokasi output, seed, dan penggunaan GPU. Folder dibuat saat notebook dijalankan.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import time

import joblib
import numpy as np
import tensorflow as tf
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import SVC

RUN_ID = "cifar10_7fitur_v1"
SEED = 30092026
VARIANT = "avg"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / VARIANT

for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Perangkat GPU:", gpus, "| Run:", RUN_ID)

I0000 00:00:1791458567.333004  148434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791458567.369529  148434 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791458568.319325  148434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Perangkat GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] | Run: cifar10_7fitur_v1


## 2. Data training
Gunakan 50.000 data training CIFAR-10 dan buat split stratified tetap 40.000/10.000.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)

train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.2,
    stratify=y_all,
    random_state=SEED,
)
train_idx = np.sort(train_idx)
val_idx = np.sort(val_idx)
assert len(train_idx) == 40000
assert len(val_idx) == 10000

split_hash = hashlib.sha256(
    train_idx.tobytes() + val_idx.tobytes()
).hexdigest()
print("Split:", len(train_idx), len(val_idx), split_hash[:16])

Split: 40000 10000 16004a1ac8a89ac9


## 3. Konversi citra
RGB asli, AVG rata-rata kanal, dan NTSC bobot luminansi menggunakan rumus yang sama pada semua notebook.

In [3]:
def prepare_images(x, variant):
    x = np.asarray(x, dtype=np.float32) / 255.0

    if variant == "rgb":
        return x
    if variant == "avg":
        return np.mean(x, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        return np.sum(x * weights, axis=-1, keepdims=True)

    raise ValueError(f"Varian tidak dikenal: {variant}")

## 4. CNN custom
Dua residual block pada tiap tahap 64/128/256. Embedding 512 dimensi. Crop berpadding dan flip aktif hanya saat training.

In [4]:
def residual_block(x, filters, stride=1):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    x = tf.keras.layers.Add()([x, shortcut])
    return tf.keras.layers.Activation("relu")(x)

def build_cnn(variant, dropout=0.3):
    channels = 3 if variant == "rgb" else 1
    inputs = tf.keras.Input((32, 32, channels))
    x = tf.keras.layers.ZeroPadding2D(4)(inputs)
    x = tf.keras.layers.RandomCrop(32, 32, seed=SEED)(x)
    x = tf.keras.layers.RandomFlip("horizontal", seed=SEED)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    for filters, first_stride in ((64, 1), (128, 2), (256, 2)):
        x = residual_block(x, filters, first_stride)
        x = residual_block(x, filters)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    embedding = tf.keras.layers.Dense(512, activation="relu", dtype="float32", name="embedding")(x)
    outputs = tf.keras.layers.Dense(10, activation="softmax", dtype="float32")(embedding)
    return tf.keras.Model(inputs, outputs)

def compile_cnn(model, lr, epochs, steps):
    schedule = tf.keras.optimizers.schedules.CosineDecay(lr, decay_steps=max(1, epochs * steps))
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=schedule, weight_decay=0.0001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

## 5. Estimator pencarian
Adaptor scikit-learn melatih CNN custom per kandidat dan menilai akurasi pada validation tetap.

In [5]:
class CNNGridEstimator(ClassifierMixin, BaseEstimator):
    def __init__(self, variant="rgb", lr=0.001, epochs=20, batch_size=64, dropout=0.3):
        self.variant, self.lr, self.epochs = variant, lr, epochs
        self.batch_size, self.dropout = batch_size, dropout
    def fit(self, X, y):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        self.model_ = build_cnn(self.variant, self.dropout)
        compile_cnn(self.model_, self.lr, self.epochs, int(np.ceil(len(X)/self.batch_size)))
        self.model_.fit(prepare_images(X, self.variant), y, batch_size=self.batch_size,
                        epochs=self.epochs, verbose=1)
        self.classes_ = np.unique(y)
        return self
    def predict(self, X):
        p = self.model_.predict(
            prepare_images(X, self.variant),
            batch_size=self.batch_size,
            verbose=1,
        )
        return np.argmax(p, axis=1)
    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

## 6. GridSearchCV CNN
Dua learning rate diuji 20 epoch; `refit=False` menjaga validation di luar fitting. Tahap ini dijalankan sebelum pelatihan CNN final.

In [6]:
fold = np.full(50000, -1, dtype=np.int8)
fold[val_idx] = 0
cnn_grid = GridSearchCV(CNNGridEstimator(variant=VARIANT),
                        {"lr": [0.001, 0.0003]}, cv=PredefinedSplit(fold),
                        refit=False, n_jobs=1, error_score="raise", verbose=1)
grid_start = time.perf_counter()
cnn_grid.fit(x_all, y_all)
cnn_grid_seconds = time.perf_counter() - grid_start
print("Terbaik:", cnn_grid.best_params_, cnn_grid.best_score_, "detik:", cnn_grid_seconds)

Fitting 1 folds for each of 2 candidates, totalling 2 fits


I0000 00:00:1791458571.105617  148434 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4139 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Epoch 1/20


I0000 00:00:1791458578.012489  148507 cuda_dnn.cc:461] Loaded cuDNN version 92600


625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 88ms/step - accuracy: 0.4338 - loss: 1.5615
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.6024 - loss: 1.1129
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.6697 - loss: 0.9372
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.7253 - loss: 0.7955
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.7613 - loss: 0.6946
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.7894 - loss: 0.6141
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8116 - loss: 0.5523
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8256 - loss: 0.5043
Epoch 9/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8482 - loss: 0.4455
Epoch 10/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8609 - loss: 0.4069
Epoch 11/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8762 - loss: 0.3638
Epoch 12/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 

## 7. Simpan hasil grid CNN
Skor ini berasal dari pelatihan singkat dan belum mewakili CNN final 150 epoch.

In [7]:
cnn_search = {"run_id": RUN_ID, "variant": VARIANT, "params": cnn_grid.best_params_,
              "validation_accuracy": float(cnn_grid.best_score_), "seconds": cnn_grid_seconds,
              "candidates": [{"params": p, "score": float(s)} for p, s in
                             zip(
                                 cnn_grid.cv_results_["params"],
                                 cnn_grid.cv_results_["mean_test_score"],
                             )]}
(OUT / "laporan" / "cnn_grid.json").write_text(json.dumps(cnn_search, indent=2), encoding="utf-8")

343

## 8. Latih CNN final
Mulai dari bobot baru, maksimal 150 epoch. Early stopping memulihkan bobot terbaik; checkpoint mencatat epoch validation terbaik. Waktu dicatat untuk mengukur biaya nyata.

In [8]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
final_cnn = build_cnn(VARIANT)
compile_cnn(final_cnn, cnn_grid.best_params_["lr"], 150, int(np.ceil(len(train_idx)/64)))
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=20,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / "model" / "best_checkpoint.keras"),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]
final_start = time.perf_counter()
history = final_cnn.fit(prepare_images(x_all[train_idx], VARIANT), y_all[train_idx],
                        validation_data=(prepare_images(x_all[val_idx], VARIANT), y_all[val_idx]),
                        batch_size=64, epochs=150, callbacks=callbacks, verbose=1)
final_seconds = time.perf_counter() - final_start
print("Epoch selesai:", len(history.epoch), "detik:", final_seconds)
print("Estimasi 1 epoch:", final_seconds / len(history.epoch), "detik; bukan jaminan total 12 jam")

Epoch 1/150
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.4381 - loss: 1.5491
Epoch 1: val_accuracy improved from None to 0.40600, saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v1/avg/model/best_checkpoint.keras

Epoch 1: finished saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v1/avg/model/best_checkpoint.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 95ms/step - accuracy: 0.4381 - loss: 1.5491 - val_accuracy: 0.4060 - val_loss: 1.6999
Epoch 2/150
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6072 - loss: 1.1009
Epoch 2: val_accuracy did not improve from 0.40600
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 95ms/step - accuracy: 0.6072 - loss: 1.1009 - val_accuracy: 0.3830 - val_loss: 1.8468
Epoch 3/150
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.6737 - loss: 0.9309
Epoch 3: val_accuracy improved from 0.40600 to 0.64980, saving model to /home/saga/anaconda3/envs/envGPU/

## 9. Simpan CNN terbaik
Muat checkpoint terbaik sebelum menyimpan model yang dipakai Extraction dan Testing.

In [9]:
final_cnn = tf.keras.models.load_model(OUT / "model" / "best_checkpoint.keras")
final_cnn.save(OUT / "model" / "cnn.keras")
cnn_history = {"seconds": final_seconds, "epochs_run": len(history.epoch),
               "history": {key: [float(v) for v in vals] for key, vals in history.history.items()},
               "best_val_accuracy": float(max(history.history["val_accuracy"])),
               "split_hash": split_hash, "seed": SEED}
(OUT / "laporan" / "cnn_training.json").write_text(
    json.dumps(cnn_history, indent=2), encoding="utf-8"
)

16152

## 10. Bebaskan memori CNN
Lepaskan model training dan hasil grid dari memori sebelum ekstraksi embedding.

In [10]:
del cnn_grid, final_cnn, history, callbacks
tf.keras.backend.clear_session()
gc.collect()

0

## 11. Muat CNN tersimpan
Ambil embedding dari layer `embedding` untuk melatih SVM pada run ini.

In [11]:
model_path = OUT / "model" / "cnn.keras"
assert model_path.exists(), "Model CNN belum tersimpan"
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()

cnn = tf.keras.models.load_model(model_path, compile=False)
extractor = tf.keras.Model(
    inputs=cnn.input,
    outputs=cnn.get_layer("embedding").output,
)
assert extractor.output_shape[-1] == 512

## 12. Ekstraksi embedding sementara
Hitung embedding training dan validation dalam batch. Hasilnya dipakai langsung oleh SVM dan tidak ditulis ke folder `fitur/`.

In [12]:
start = time.perf_counter()
features = extractor.predict(
    prepare_images(x_all, VARIANT),
    batch_size=128,
    verbose=1,
).astype(np.float32)

assert features.shape == (50000, 512)
assert np.isfinite(features).all()
print("Durasi ekstraksi sementara (detik):", time.perf_counter() - start)

391/391 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step
Durasi ekstraksi sementara (detik): 22.38829610800167


## 13. Bebaskan memori model ekstraksi
Lepaskan CNN dari memori; array `features` tetap tersedia untuk pencarian SVM.

In [13]:
del cnn, extractor
tf.keras.backend.clear_session()
gc.collect()

0

## 14. Periksa embedding sementara
Pastikan bentuk fitur, split, dan identitas model CNN sesuai sebelum melatih SVM.

In [14]:
assert features.shape == (len(y_all), 512)
assert len(train_idx) == 40000
assert len(val_idx) == 10000
assert np.isfinite(features).all()
assert hashlib.sha256(model_path.read_bytes()).hexdigest() == model_sha

## 15. GridSearchCV SVM
Scaler dan normalisasi di-fit ulang di dalam tiap kandidat pada 40.000 training; validation hanya untuk skor.

In [15]:
fold = np.full(len(y_all), -1, dtype=np.int8)
fold[val_idx] = 0

svm_pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("norm", Normalizer()),
        ("svc", SVC(kernel="rbf", cache_size=512, random_state=SEED, verbose=1)),
    ]
)
svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid={
        "svc__C": [1, 10],
        "svc__gamma": ["scale", 0.001],
    },
    cv=PredefinedSplit(fold),
    refit=False,
    n_jobs=1,
    error_score="raise",
    verbose=1,
)

svm_start = time.perf_counter()
svm_grid.fit(features, y_all)
svm_seconds = time.perf_counter() - svm_start
print("Parameter terbaik:", svm_grid.best_params_)
print("Akurasi validation:", svm_grid.best_score_)
print("Durasi pencarian (detik):", svm_seconds)

Fitting 1 folds for each of 4 candidates, totalling 4 fits
[LibSVM]*
optimization finished, #iter = 130
obj = -6.044135, rho = 0.045568
nSV = 51, nBSV = 3
*
optimization finished, #iter = 113
obj = -7.566741, rho = 0.036686
nSV = 38, nBSV = 4
*
optimization finished, #iter = 87
obj = -7.561342, rho = 0.181420
nSV = 40, nBSV = 5
*
optimization finished, #iter = 96
obj = -6.849618, rho = 0.002378
nSV = 38, nBSV = 5
*
optimization finished, #iter = 82
obj = -5.778200, rho = -0.023191
nSV = 40, nBSV = 2
*
optimization finished, #iter = 93
obj = -6.992279, rho = 0.051870
nSV = 41, nBSV = 5
*
optimization finished, #iter = 104
obj = -6.536456, rho = 0.030206
nSV = 41, nBSV = 2
*
optimization finished, #iter = 62
obj = -8.759029, rho = -0.063433
nSV = 33, nBSV = 7
*
optimization finished, #iter = 111
obj = -7.217036, rho = 0.115617
nSV = 46, nBSV = 4
*
optimization finished, #iter = 75
obj = -5.552754, rho = -0.024784
nSV = 35, nBSV = 2
*
optimization finished, #iter = 73
obj = -7.360918, rho

## 16. Fit SVM terpilih
Fit hanya pada 40.000 training dengan parameter dari grid; validation tetap terpisah.

In [16]:
final_svm = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("norm", Normalizer()),
        (
            "svc",
            SVC(
                kernel="rbf",
                C=svm_grid.best_params_["svc__C"],
                gamma=svm_grid.best_params_["svc__gamma"],
                cache_size=512,
                random_state=SEED,
                verbose=1,
            ),
        ),
    ]
)
final_svm.fit(features[train_idx], y_all[train_idx])
final_val_accuracy = float(
    accuracy_score(y_all[val_idx], final_svm.predict(features[val_idx]))
)
print("Akurasi validation SVM final:", final_val_accuracy)

[LibSVM]*
optimization finished, #iter = 131
obj = -6.102092, rho = 0.044192
nSV = 53, nBSV = 0
*
optimization finished, #iter = 116
obj = -7.637209, rho = 0.046446
nSV = 39, nBSV = 0
*
optimization finished, #iter = 113
obj = -7.575966, rho = 0.175756
nSV = 38, nBSV = 0
*
optimization finished, #iter = 97
obj = -6.998842, rho = 0.002250
nSV = 30, nBSV = 0
*
optimization finished, #iter = 84
obj = -5.889506, rho = -0.035795
nSV = 37, nBSV = 0
*
optimization finished, #iter = 93
obj = -7.506543, rho = 0.062062
nSV = 33, nBSV = 0
*
optimization finished, #iter = 126
obj = -6.553029, rho = 0.028533
nSV = 42, nBSV = 0
*
optimization finished, #iter = 86
obj = -9.180256, rho = -0.046500
nSV = 30, nBSV = 0
*
optimization finished, #iter = 100
obj = -7.552213, rho = 0.116212
nSV = 39, nBSV = 0
*
optimization finished, #iter = 80
obj = -5.554741, rho = -0.024915
nSV = 37, nBSV = 0
*
optimization finished, #iter = 90
obj = -7.534588, rho = 0.144380
nSV = 35, nBSV = 0
*
optimization finished, #i

## 17. Simpan SVM dan laporan
Model dan metadata identitas CNN disimpan untuk verifikasi di Testing.

In [17]:
joblib.dump({"pipeline": final_svm, "run_id": RUN_ID, "variant": VARIANT,
             "split_hash": split_hash, "model_sha": model_sha}, OUT / "model" / "svm.joblib")
svm_report = {
    "params": svm_grid.best_params_,
    "grid_validation_accuracy": float(svm_grid.best_score_),
              "final_validation_accuracy": final_val_accuracy, "seconds": svm_seconds,
              "candidates": [{"params": p, "score": float(s)} for p, s in
                             zip(
                                 svm_grid.cv_results_["params"],
                                 svm_grid.cv_results_["mean_test_score"],
                             )]}
(OUT / "laporan" / "svm_grid.json").write_text(json.dumps(svm_report, indent=2), encoding="utf-8")

644